# Scoring logos with TRIBE v2

[TRIBE v2](https://huggingface.co/facebook/tribev2) predicts fMRI brain responses to naturalistic stimuli (video/audio/text). Here we treat each logo as a silent visual stimulus, get TRIBE v2's predicted brain-response vector for it, then use PCA to place all logos in a 2D space to see how they relate to each other.

In [1]:
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(".env")
import os

if os.environ.get("HUGGING_FACE_TOKEN"):
    login(token=os.environ["HUGGING_FACE_TOKEN"])

LOGOS_DIR = Path("logos")
CACHE_FOLDER = Path("./cache")
logo_paths = sorted(LOGOS_DIR.glob("*.png")) + sorted(LOGOS_DIR.glob("*.jpeg"))
logo_paths

[PosixPath('logos/anthropic.png'),
 PosixPath('logos/carulla.png'),
 PosixPath('logos/exito.png'),
 PosixPath('logos/truora.png'),
 PosixPath('logos/images.jpeg'),
 PosixPath('logos/openai.jpeg')]

## Load TRIBE v2

Downloads the checkpoint from Hugging Face on first run (~1GB).

In [2]:
from tribev2.demo_utils import TribeModel

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    config_update={
        "data.text_feature.device": "cpu",
        "data.audio_feature.device": "cpu",
        "data.image_feature.image.device": "cpu",
        "data.video_feature.image.device": "cpu",
    },
)

/Users/juandavidramirezjimenez/Documents/platanus-hack-26-co-team-29/.venv/lib/python3.11/site-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-08-22 02:23:01 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
2026-08-22 02:23:01 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /Users/juandavidramirezjimenez/.cache/huggingface/hub/models--facebook-

## Turn each logo into a short silent clip

TRIBE v2 only accepts video/audio/text files, not static images, so each logo is rendered as a short looping clip (no audio track).

In [3]:
from moviepy import ImageClip

CLIP_DURATION = 1
video_paths = {}
for logo_path in logo_paths:
    video_path = CACHE_FOLDER / f"{logo_path.stem}.mp4"
    clip = ImageClip(str(logo_path), duration=CLIP_DURATION).resized(height=256)
    clip.write_videofile(str(video_path), codec="libx264", audio=False, fps=8, logger=None)
    video_paths[logo_path.stem] = video_path
video_paths

{'anthropic': PosixPath('cache/anthropic.mp4'),
 'carulla': PosixPath('cache/carulla.mp4'),
 'exito': PosixPath('cache/exito.mp4'),
 'truora': PosixPath('cache/truora.mp4'),
 'images': PosixPath('cache/images.mp4'),
 'openai': PosixPath('cache/openai.mp4')}

## Score each logo

For each clip we build the events dataframe, run `model.predict`, and average the predicted brain response over time to get one embedding vector per logo.

In [ ]:
embeddings = {}
for name, video_path in video_paths.items():
    print(f"Scoring {name}...")
    events = model.get_events_dataframe(video_path=video_path)
    preds, segments = model.predict(events=events, verbose=False)
    embeddings[name] = preds.mean(axis=0)

names = list(embeddings.keys())
X = np.stack([embeddings[name] for name in names])
X.shape

Scoring anthropic...


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 31.50it/s]
Extracting words from audio: 0it [00:00, ?it/s]
No transcripts found, skipping
2026-08-22 02:23:02 - INFO - neuralset.events.transforms.text:56 - No Word events found, skipping
2026-08-22 02:23:02 - INFO - neuralset.events.transforms.text:175 - No Word events found, skipping
Add context to words: 0it [00:00, ?it/s]
[02:23:02 WARNING] Removing extractor audio as there are no corresponding events
[02:23:02 WARNING] Removing extractor text as there are no corresponding events
[02:23:02 INFO] Preparing extractor: video
[02:23:02 INFO] Preparing extractor: subject_id
2026-08-22 02:23:02 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:23:02 INFO] Building dataloader for split all
/Users/juandavidramirezjimenez/Documents/platanus-hack-26-co-team-29/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:624: UserWarning: T

Scoring carulla...


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 31.60it/s]
Extracting words from audio: 0it [00:00, ?it/s]
No transcripts found, skipping
2026-08-22 02:23:10 - INFO - neuralset.events.transforms.text:56 - No Word events found, skipping
2026-08-22 02:23:10 - INFO - neuralset.events.transforms.text:175 - No Word events found, skipping
Add context to words: 0it [00:00, ?it/s]
[02:23:10 WARNING] Removing extractor audio as there are no corresponding events
[02:23:10 WARNING] Removing extractor text as there are no corresponding events
[02:23:10 INFO] Preparing extractor: video


Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

...

2026-08-22 02:25:38 - DEBUG - neuralset.extractors.video:277 - Loaded Video (duration 1.0s at 8.0fps, shape (192, 256)):
cache/exito.mp4
Encoding video: 100%|██████████| 2/2 [02:19<00:00, 69.92s/it]
[02:27:58 INFO] Preparing extractor: subject_id
2026-08-22 02:27:58 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:27:59 INFO] Building dataloader for split all
/Users/juandavidramirezjimenez/Documents/platanus-hack-26-co-team-29/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 10 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.war

Scoring truora...


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 38.82it/s]
Extracting words from audio: 0it [00:00, ?it/s]
No transcripts found, skipping
2026-08-22 02:28:07 - INFO - neuralset.events.transforms.text:56 - No Word events found, skipping
2026-08-22 02:28:07 - INFO - neuralset.events.transforms.text:175 - No Word events found, skipping
Add context to words: 0it [00:00, ?it/s]
[02:28:07 WARNING] Removing extractor audio as there are no corresponding events
[02:28:07 WARNING] Removing extractor text as there are no corresponding events
[02:28:07 INFO] Preparing extractor: video


Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

2026-08-22 02:28:08 - DEBUG - neuralset.extractors.video:277 - Loaded Video (duration 1.0s at 8.0fps, shape (437, 256)):
cache/truora.mp4
Encoding video:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:

names = list(embeddings.keys())
X = np.stack([embeddings[name] for name in names])
X.shape

(1, 20484)

## PCA to 2D and plot

In [13]:
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from PIL import Image
from sklearn.decomposition import PCA

coords = PCA(n_components=2).fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(coords[:, 0], coords[:, 1], s=0)
for (x, y), logo_path in zip(coords, logo_paths):
    img = Image.open(logo_path).convert("RGBA")
    ab = AnnotationBbox(OffsetImage(img, zoom=0.15), (x, y), frameon=False)
    ax.add_artist(ab)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Logos placed by TRIBE v2 predicted brain response (PCA)")
plt.show()

ValueError: n_components=2 must be between 0 and min(n_samples, n_features)=1 with svd_solver='full'